# Tarea 3 — PyTorch

**Nombre del integrante:** Diego Moreno

**Cédula del integrante:** 32226501

> Enunciado, rúbrica y nombres de los archivos `.pth` a entregar: ver `README.md` de esta carpeta.
> Antes del commit: *Restart & Run All* debe terminar sin errores y quedar en el menú.

## 1. Imports y constantes

In [ ]:
import copy
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NOTEBOOK_NUMBER = 1
DATA_DIR = "./data"
INPUT_SIZE = 28 * 28
SEED = 42

CLASS_NAMES = {
    "original": [
        "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
        "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
    ],
    "simplified": ["Top", "Bottom", "Footwear", "Bag"],
}

# Fashion MNIST label -> simplified label (0 = Top, 1 = Bottom, 2 = Footwear, 3 = Bag)
SIMPLIFIED_LABEL = torch.tensor([0, 1, 0, 1, 0, 2, 0, 2, 3, 2])

DEFAULT_OPTIMIZATIONS = {
    "optimizer": "adam",        # "adam" or "sgd" (SGD uses momentum 0.9)
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 128,
    "early_stopping": True,
    "patience": 3,              # epochs without validation improvement before stopping
    "batch_norm": True,
    "dropout": 0.2,
}

print(f"Device: {DEVICE}")

## 2. Datos (Fashion MNIST original y clases simplificadas)

Las imágenes se cargan una sola vez en memoria como tensores en `[0, 1]`, que es exactamente lo que
produce `transforms.ToTensor()` (sin `Normalize`). Del conjunto de entrenamiento se separa un 10%
como **validación**, que se usa para el *early stopping*; el conjunto de prueba solo se usa en
"Probar".

Clasificación simplificada: **Top** = T-shirt, Pullover, Coat, Shirt · **Bottom** = Trouser, Dress ·
**Footwear** = Sandal, Sneaker, Ankle boot · **Bag** = Bag.

In [ ]:
def load_split(train):
    """Download (if needed) one Fashion MNIST split and return (images, labels) tensors."""
    dataset = torchvision.datasets.FashionMNIST(DATA_DIR, train=train, download=True)
    # Same values as transforms.ToTensor(): uint8 pixels scaled to [0, 1], shape (N, 1, 28, 28)
    images = dataset.data.unsqueeze(1).float() / 255.0
    return images, dataset.targets.clone()


TRAIN_IMAGES, TRAIN_LABELS = load_split(train=True)
TEST_IMAGES, TEST_LABELS = load_split(train=False)

# Fixed train/validation split so results are reproducible across runs
_permutation = torch.randperm(len(TRAIN_LABELS), generator=torch.Generator().manual_seed(SEED))
VALIDATION_SIZE = len(TRAIN_LABELS) // 10
VALIDATION_INDEX = _permutation[:VALIDATION_SIZE]
TRAIN_INDEX = _permutation[VALIDATION_SIZE:]


def to_labels(labels, classification):
    """Map native Fashion MNIST labels to the requested classification."""
    return SIMPLIFIED_LABEL[labels] if classification == "simplified" else labels


def make_loaders(classification, batch_size):
    """Return (train, validation, test) DataLoaders for the given classification."""
    train_labels = to_labels(TRAIN_LABELS, classification)
    test_labels = to_labels(TEST_LABELS, classification)
    train = TensorDataset(TRAIN_IMAGES[TRAIN_INDEX], train_labels[TRAIN_INDEX])
    validation = TensorDataset(TRAIN_IMAGES[VALIDATION_INDEX], train_labels[VALIDATION_INDEX])
    test = TensorDataset(TEST_IMAGES, test_labels)
    return (
        DataLoader(train, batch_size=batch_size, shuffle=True),
        DataLoader(validation, batch_size=1024),
        DataLoader(test, batch_size=1024),
    )


print(f"Train: {len(TRAIN_INDEX)} | Validation: {VALIDATION_SIZE} | Test: {len(TEST_LABELS)}")

## 3. Modelos (rectangular y convolucional)

- **Rectangular:** todas las capas ocultas tienen el mismo número de neuronas.
- **Convolucional (embudo):** la capa oculta `i` de `L` tiene
  `round(784 + (salida - 784) * i / (L + 1))` neuronas, de modo que cada capa disminuye en la
  misma cantidad. Con 10 clases y `L = 2`: `784 → 526 → 268 → 10`.

Cada capa oculta es `Linear → [BatchNorm1d] → ReLU → [Dropout]`; el modelo es un `nn.Sequential`
que empieza en `nn.Flatten()`.

In [ ]:
def num_classes_for(classification):
    return len(CLASS_NAMES[classification])


def hidden_layer_sizes(network_type, hidden_layers, neurons_per_layer, num_classes):
    """Neurons of each hidden layer for a rectangular or convolutional (funnel) network."""
    if network_type == "rectangular":
        return [neurons_per_layer] * hidden_layers
    step = (num_classes - INPUT_SIZE) / (hidden_layers + 1)
    return [round(INPUT_SIZE + step * i) for i in range(1, hidden_layers + 1)]


def build_model(layer_sizes, num_classes, batch_norm, dropout):
    """Build the nn.Sequential network described by the hidden layer sizes."""
    layers = [nn.Flatten()]
    in_features = INPUT_SIZE
    for out_features in layer_sizes:
        layers.append(nn.Linear(in_features, out_features))
        if batch_norm:
            layers.append(nn.BatchNorm1d(out_features))
        layers.append(nn.ReLU())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        in_features = out_features
    layers.append(nn.Linear(in_features, num_classes))
    return nn.Sequential(*layers).to(DEVICE)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

## 4. Entrenar, probar, guardar y cargar

- **Entrenar:** entropía cruzada con Adam o SGD (configurable), *learning rate*, *weight decay* y
  *early stopping* sobre el conjunto de validación (se restauran los mejores pesos). Al terminar se
  grafica la pérdida por época.
- **Probar:** matriz de confusión, *accuracy* y *precision* / *recall* por clase y promedio (macro).
- **Guardar / cargar:** `{"state_dict", "hyperparameters"}`; al cargar, la arquitectura se
  reconstruye a partir de las formas del `state_dict`.

In [ ]:
def make_optimizer(model, optimizations):
    lr = optimizations["learning_rate"]
    weight_decay = optimizations["weight_decay"]
    if optimizations["optimizer"] == "sgd":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)


def run_epoch(model, loader, loss_fn, optimizer=None):
    """One pass over the loader. Trains when an optimizer is given. Returns (loss, accuracy)."""
    model.train(optimizer is not None)
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(optimizer is not None):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            if optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += len(labels)
    return total_loss / total, correct / total


def train(model, hyperparameters, epochs, verbose=True):
    """Train for up to `epochs` epochs. Returns per-epoch train and validation losses."""
    optimizations = hyperparameters["optimizations"]
    train_loader, validation_loader, _ = make_loaders(
        hyperparameters["classification"], optimizations["batch_size"]
    )
    loss_fn = nn.CrossEntropyLoss()
    optimizer = make_optimizer(model, optimizations)

    history = {"train": [], "validation": []}
    best_loss, best_state, epochs_without_improvement = float("inf"), None, 0
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = run_epoch(model, validation_loader, loss_fn)
        history["train"].append(train_loss)
        history["validation"].append(val_loss)
        hyperparameters["epochs"] += 1
        if verbose:
            print(f"Epoch {epoch:3d}/{epochs}  train loss {train_loss:.4f} acc {train_acc:.2%}  "
                  f"| val loss {val_loss:.4f} acc {val_acc:.2%}")

        if val_loss < best_loss:
            best_loss, best_state, epochs_without_improvement = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            epochs_without_improvement += 1
        if optimizations["early_stopping"] and epochs_without_improvement >= optimizations["patience"]:
            if verbose:
                print(f"Early stopping: {epochs_without_improvement} epochs without validation "
                      f"improvement. Restoring best weights (val loss {best_loss:.4f}).")
            model.load_state_dict(best_state)
            break
    return history


def plot_loss(history):
    epochs = range(1, len(history["train"]) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, history["train"], marker="o", label="Entrenamiento")
    plt.plot(epochs, history["validation"], marker="s", label="Validación")
    plt.title("Pérdida (loss) por época")
    plt.xlabel("Época")
    plt.ylabel("Pérdida (entropía cruzada)")
    plt.xticks(list(epochs))
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()


@torch.no_grad()
def evaluate(model, classification):
    """Predict the test set. Returns the confusion matrix (rows = true, columns = predicted)."""
    model.eval()
    _, _, test_loader = make_loaders(classification, batch_size=1024)
    n = num_classes_for(classification)
    confusion = torch.zeros(n, n, dtype=torch.long)
    for images, labels in test_loader:
        predictions = model(images.to(DEVICE)).argmax(1).cpu()
        confusion += torch.bincount(labels * n + predictions, minlength=n * n).reshape(n, n)
    return confusion.numpy()


def metrics_from_confusion(confusion):
    true_positives = np.diag(confusion)
    predicted = confusion.sum(axis=0)
    actual = confusion.sum(axis=1)
    precision = np.divide(true_positives, predicted, out=np.zeros(len(predicted)), where=predicted > 0)
    recall = np.divide(true_positives, actual, out=np.zeros(len(actual)), where=actual > 0)
    accuracy = true_positives.sum() / confusion.sum()
    return accuracy, precision, recall


def report(model, classification):
    """Print accuracy / precision / recall and plot the confusion matrix of the test set."""
    confusion = evaluate(model, classification)
    accuracy, precision, recall = metrics_from_confusion(confusion)
    names = CLASS_NAMES[classification]

    print(f"\nAccuracy:           {accuracy:.2%}")
    print(f"Precision (macro):  {precision.mean():.2%}")
    print(f"Recall (macro):     {recall.mean():.2%}\n")
    print(f"{'Clase':<14}{'Precision':>10}{'Recall':>10}")
    for name, p, r in zip(names, precision, recall):
        print(f"{name:<14}{p:>10.2%}{r:>10.2%}")

    size = 4 + 0.5 * len(names)
    fig, ax = plt.subplots(figsize=(size, size))
    ax.imshow(confusion, cmap="Blues")
    ax.set_xticks(range(len(names)), names, rotation=45, ha="right")
    ax.set_yticks(range(len(names)), names)
    threshold = confusion.max() / 2
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, confusion[i, j], ha="center", va="center",
                    color="white" if confusion[i, j] > threshold else "black", fontsize=8)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Clase real")
    ax.set_title(f"Matriz de confusión — accuracy {accuracy:.2%}")
    fig.tight_layout()
    plt.show()
    return accuracy


def save_network(model, hyperparameters, path):
    torch.save({"state_dict": model.state_dict(), "hyperparameters": hyperparameters}, path)
    print(f"Saved: {path}")


def model_from_state_dict(state_dict, dropout):
    """Rebuild the Sequential network from the shapes of its state_dict."""
    linear_keys = sorted(
        (k for k, v in state_dict.items() if k.endswith(".weight") and v.dim() == 2),
        key=lambda k: int(k.split(".")[0]),
    )
    layer_sizes = [state_dict[k].shape[0] for k in linear_keys]
    batch_norm = any(k.endswith("running_mean") for k in state_dict)
    if len(linear_keys) > 1:
        # Each hidden block is Linear, [BatchNorm], ReLU, [Dropout]: its stride tells if Dropout exists
        stride = int(linear_keys[1].split(".")[0]) - int(linear_keys[0].split(".")[0])
        has_dropout = stride == 2 + batch_norm + 1
        dropout = (dropout or 0.2) if has_dropout else 0.0
    model = build_model(layer_sizes[:-1], layer_sizes[-1], batch_norm, dropout)
    model.load_state_dict(state_dict)
    return model


def load_network(path):
    checkpoint = torch.load(path, map_location=DEVICE)
    hyperparameters = checkpoint.get("hyperparameters", {})
    optimizations = {**DEFAULT_OPTIMIZATIONS, **hyperparameters.get("optimizations", {})}
    model = model_from_state_dict(checkpoint["state_dict"], optimizations["dropout"])

    sizes = [m.out_features for m in model if isinstance(m, nn.Linear)]
    optimizations["batch_norm"] = any(isinstance(m, nn.BatchNorm1d) for m in model)
    # The shapes of the state_dict take precedence over whatever the file claims
    hyperparameters = {
        "network_type": "rectangular",
        "epochs": 0,
        **hyperparameters,
        "classification": "original" if sizes[-1] == 10 else "simplified",
        "hidden_layers": len(sizes) - 1,
        "neurons_per_layer": sizes[:-1],
        "optimizations": optimizations,
    }
    return model, hyperparameters


def default_filename(hyperparameters):
    return (f"pytorch_fashion_{NOTEBOOK_NUMBER}_{hyperparameters['classification']}_"
            f"{hyperparameters['network_type']}.pth")


def describe(model, hyperparameters):
    sizes = " → ".join(str(n) for n in [INPUT_SIZE, *hyperparameters["neurons_per_layer"],
                                        num_classes_for(hyperparameters["classification"])])
    o = hyperparameters["optimizations"]
    print(f"Red {hyperparameters['network_type']} | clasificación {hyperparameters['classification']} | "
          f"{hyperparameters['hidden_layers']} capas ocultas | épocas acumuladas {hyperparameters['epochs']}")
    print(f"Arquitectura: {sizes}  ({count_parameters(model):,} parámetros)")
    print(f"Optimizaciones: optimizador {o['optimizer']}, lr {o['learning_rate']}, weight decay "
          f"{o['weight_decay']}, batch {o['batch_size']}, batch norm {'sí' if o['batch_norm'] else 'no'}, "
          f"dropout {o['dropout']}, early stopping "
          f"{'sí (paciencia ' + str(o['patience']) + ')' if o['early_stopping'] else 'no'}")

## 5. Redes entregadas

Las cuatro redes se crearon con este mismo programa. Se probaron varias combinaciones de capas
ocultas y neuronas por capa, buscando buena *accuracy* con pocas neuronas y pocas épocas
(Adam, lr `0.001`, weight decay `1e-4`, batch norm, dropout `0.2`, early stopping con paciencia 3;
máximo 20 épocas):

| Clasificación | Tipo | Capas ocultas | Arquitectura | Parámetros | Épocas | Accuracy (prueba) |
|---|---|---|---|---|---|---|
| Original (10) | rectangular | 1 | 784 → 128 → 10 | 102,026 | 14 | 88.01% |
| Original (10) | rectangular | 1 | 784 → 256 → 10 | 204,042 | 10 | 88.18% |
| Original (10) | rectangular | 2 | **784 → 128 → 128 → 10** | 118,794 | 16 | **88.80%** |
| Original (10) | rectangular | 2 | 784 → 256 → 256 → 10 | 270,346 | 15 | 88.71% |
| Original (10) | rectangular | 2 | 784 → 512 → 512 → 10 | 671,754 | 10 | 88.15% |
| Original (10) | convolutional | 1 | 784 → 397 → 10 | 316,419 | 11 | 88.42% |
| Original (10) | convolutional | 2 | **784 → 526 → 268 → 10** | 558,424 | 16 | **89.04%** |
| Original (10) | convolutional | 3 | 784 → 590 → 397 → 204 → 10 | 783,401 | 8 | 87.93% |
| Simplificada (4) | rectangular | 1 | **784 → 64 → 4** | 50,628 | 9 | **97.01%** |
| Simplificada (4) | rectangular | 1 | 784 → 128 → 4 | 101,252 | 8 | 97.17% |
| Simplificada (4) | rectangular | 2 | 784 → 128 → 128 → 4 | 118,020 | 7 | 97.31% |
| Simplificada (4) | rectangular | 2 | 784 → 512 → 512 → 4 | 668,676 | 4 | 96.73% |
| Simplificada (4) | convolutional | 1 | **784 → 394 → 4** | 311,658 | 7 | **97.07%** |
| Simplificada (4) | convolutional | 2 | 784 → 524 → 264 → 4 | 552,576 | 11 | 97.23% |
| Simplificada (4) | convolutional | 3 | 784 → 589 → 394 → 199 → 4 | 776,594 | 8 | 96.93% |

En **negrita** las redes entregadas (con early stopping el entrenamiento se detiene solo; las
épocas incluyen las de paciencia, y se restauran los mejores pesos de validación):

| Archivo | Red | Accuracy (prueba) |
|---|---|---|
| `pytorch_fashion_1_original_rectangular.pth` | 784 → 128 → 128 → 10 | 88.80% |
| `pytorch_fashion_1_original_convolutional.pth` | 784 → 526 → 268 → 10 | 89.04% |
| `pytorch_fashion_1_simplified_rectangular.pth` | 784 → 64 → 4 | 97.01% |
| `pytorch_fashion_1_simplified_convolutional.pth` | 784 → 394 → 4 | 97.07% |

Observaciones: más neuronas no mejoran la red rectangular (512×2 queda por debajo de 128×2 y
sobreajusta antes); en la clasificación simplificada basta **una** capa oculta pequeña, porque
casi todo el error está entre Top y Bottom (vestidos vs. camisas). Para la convolucional, 2 capas
es lo mejor con 10 clases y 1 capa con 4 clases; una tercera capa empeora el resultado.

## 6. Menú

Opciones: crear o cargar una red, entrenarla (muestra la pérdida por época), probarla (muestra la
matriz de confusión y las métricas), guardarla y configurar las optimizaciones de entrenamiento.
El menú vuelve a aparecer después de cada acción.

In [ ]:
def ask(prompt, default):
    answer = input(f"{prompt} [{default}]: ").strip()
    return answer or str(default)


def ask_int(prompt, default, minimum=1):
    while True:
        try:
            value = int(ask(prompt, default))
            if value >= minimum:
                return value
        except ValueError:
            pass
        print(f"  Ingresa un entero mayor o igual a {minimum}.")


def ask_float(prompt, default, minimum=0.0, maximum=float("inf")):
    while True:
        try:
            value = float(ask(prompt, default))
            if minimum <= value <= maximum:
                return value
        except ValueError:
            pass
        print(f"  Ingresa un número entre {minimum} y {maximum}.")


def ask_option(prompt, options, default):
    """options: {key typed by the user: value}. Returns the selected value."""
    while True:
        answer = ask(prompt, default).lower()
        if answer in options:
            return options[answer]
        print(f"  Opciones válidas: {', '.join(options)}.")


def ask_yes_no(prompt, default):
    return ask_option(f"{prompt} (s/n)", {"s": True, "n": False}, "s" if default else "n")


def ask_architecture_options(optimizations):
    optimizations["batch_norm"] = ask_yes_no("¿Usar batch normalization?", optimizations["batch_norm"])
    optimizations["dropout"] = ask_float("Dropout (0 = sin dropout)", optimizations["dropout"], 0.0, 0.9)


def ask_training_options(optimizations):
    optimizations["optimizer"] = ask_option("Optimizador (adam/sgd)", {"adam": "adam", "sgd": "sgd"},
                                            optimizations["optimizer"])
    optimizations["learning_rate"] = ask_float("Learning rate", optimizations["learning_rate"], 1e-6, 10.0)
    optimizations["weight_decay"] = ask_float("Weight decay (0 = sin)", optimizations["weight_decay"], 0.0, 1.0)
    optimizations["batch_size"] = ask_int("Tamaño de batch", optimizations["batch_size"])
    optimizations["early_stopping"] = ask_yes_no("¿Usar early stopping?", optimizations["early_stopping"])
    if optimizations["early_stopping"]:
        optimizations["patience"] = ask_int("Paciencia (épocas sin mejorar)", optimizations["patience"])


def create_network():
    print("\n--- Crear red ---")
    classification = ask_option("Clasificación: 1 = original (10 clases), 2 = simplificada (4 clases)",
                                {"1": "original", "2": "simplified"}, "1")
    network_type = ask_option("Tipo: 1 = rectangular, 2 = convolucional",
                              {"1": "rectangular", "2": "convolutional"}, "1")
    hidden_layers = ask_int("Número de capas ocultas", 2)
    neurons = ask_int("Neuronas por capa", 512) if network_type == "rectangular" else None
    optimizations = dict(DEFAULT_OPTIMIZATIONS)
    ask_architecture_options(optimizations)
    if ask_yes_no("¿Cambiar las opciones de entrenamiento (optimizador, lr, early stopping...)?", False):
        ask_training_options(optimizations)

    num_classes = num_classes_for(classification)
    sizes = hidden_layer_sizes(network_type, hidden_layers, neurons, num_classes)
    model = build_model(sizes, num_classes, optimizations["batch_norm"], optimizations["dropout"])
    hyperparameters = {
        "network_type": network_type,
        "classification": classification,
        "hidden_layers": hidden_layers,
        "neurons_per_layer": sizes,
        "epochs": 0,
        "optimizations": optimizations,
    }
    print("\nRed creada:")
    describe(model, hyperparameters)
    return model, hyperparameters


def load_from_file():
    print("\n--- Cargar red ---")
    files = sorted(f for f in os.listdir(".") if f.endswith(".pth"))
    for i, name in enumerate(files, 1):
        print(f"  {i}. {name}")
    answer = input("Número o ruta del archivo .pth: ").strip()
    path = files[int(answer) - 1] if answer.isdigit() and 1 <= int(answer) <= len(files) else answer
    if not os.path.isfile(path):
        print(f"No existe el archivo '{path}'.")
        return None, None
    model, hyperparameters = load_network(path)
    print(f"\nRed cargada desde {path}:")
    describe(model, hyperparameters)
    return model, hyperparameters


def main_menu():
    model, hyperparameters = None, None
    while True:
        print("\n" + "=" * 50)
        if model is None:
            print("No hay ninguna red cargada.")
        else:
            print(f"Red actual: {hyperparameters['network_type']}, {hyperparameters['classification']}, "
                  f"{hyperparameters['epochs']} épocas")
        print("=" * 50)
        print("1. Crear una red nueva")
        print("2. Cargar una red de un archivo .pth")
        print("3. Entrenar la red")
        print("4. Probar la red con el conjunto de prueba")
        print("5. Guardar la red")
        print("6. Configurar optimizaciones de entrenamiento")
        print("7. Ver la red actual")
        print("0. Salir")
        choice = input("Opción: ").strip()

        if choice == "0":
            print("¡Hasta luego!")
            break
        if choice == "1":
            model, hyperparameters = create_network()
        elif choice == "2":
            loaded = load_from_file()
            if loaded[0] is not None:
                model, hyperparameters = loaded
        elif choice in ("3", "4", "5", "6", "7") and model is None:
            print("Primero crea (1) o carga (2) una red.")
        elif choice == "3":
            epochs = ask_int("Número de épocas", 10)
            plot_loss(train(model, hyperparameters, epochs))
        elif choice == "4":
            report(model, hyperparameters["classification"])
        elif choice == "5":
            path = ask("Nombre del archivo", default_filename(hyperparameters))
            save_network(model, hyperparameters, path if path.endswith(".pth") else path + ".pth")
        elif choice == "6":
            print("(Batch norm y dropout son parte de la arquitectura: se eligen al crear la red.)")
            ask_training_options(hyperparameters["optimizations"])
            describe(model, hyperparameters)
        elif choice == "7":
            describe(model, hyperparameters)
        else:
            print("Opción no válida.")


main_menu()